# 01 — Exploratory Data Analysis

Danish News Sentiment Monitor. Explores topic distributions over time, sentiment by source, top topic terms, and source-tone correlation.

Reads directly from Neon via the shared engine in `src/db.py`. Run the pipeline (`python src/fetch.py`) first so the table is populated.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Make src/ importable from the notebooks/ directory.
sys.path.insert(0, str(Path.cwd().parent / "src"))
import db

df = pd.read_sql("SELECT * FROM articles", db.get_engine())
df["published_at"] = pd.to_datetime(df["published_at"], utc=True)
print(df.shape)
df.head()

## Pipeline completeness
How many rows have made it through each stage?

In [ ]:
df[["title", "sentiment_score", "topic_id"]].notna().sum()

## Sentiment by source

In [ ]:
(df.dropna(subset=["sentiment_score"])
   .groupby("source")["sentiment_score"]
   .agg(["count", "mean", "std"])
   .sort_values("mean"))

## Topic distribution over time

In [ ]:
ts = (df.dropna(subset=["topic_id", "published_at"])
        .assign(day=lambda d: d["published_at"].dt.date)
        .groupby(["day", "topic_id"]).size().unstack(fill_value=0))
ts.plot(kind="area", figsize=(11, 5), title="Articles per topic per day")

## Top terms per topic
Loads the fitted LDA model saved by `src/topics.py`.

In [ ]:
import topics

model = topics._load_model()
if model is None:
    print("No saved model — run `python src/topics.py` first.")
else:
    vec, lda = model
    for tid, terms in topics.top_terms(vec, lda).items():
        print(f"Topic {tid}: {', '.join(terms)}")

## Source-tone correlation
Do sources agree on sentiment within the same topic?

In [ ]:
pivot = (df.dropna(subset=["topic_id", "sentiment_score"])
           .pivot_table(index="topic_id", columns="source",
                        values="sentiment_score", aggfunc="mean"))
display(pivot.round(3))
pivot.corr().round(2)